# SAM3 playground

Scratchpad for trying out [Segment Anything 3](https://github.com/facebookresearch/sam3)
before turning the useful bits into the `sam3Toolkit` package.

**One-time setup** (from the repo root):

1. `uv sync --all-extras`: installs the SAM3 stack (CUDA 12.8 PyTorch) and Jupyter.
2. Request access to the gated checkpoint at <https://huggingface.co/facebook/sam3>
   (approval is manual and can take a while).
3. `uv run hf auth login` and paste a Hugging Face *read* token.
4. Open this notebook with the `.venv` kernel (VS Code), or run `uv run jupyter lab`.

The first model load downloads the checkpoint (~3.5 GB) into the Hugging Face cache.

## Environment

In [ ]:
# Corporate TLS inspection: trust the OS certificate store (needed for Hugging Face
# downloads). Must run before anything opens an HTTPS connection.
import truststore

truststore.inject_into_ssl()

In [ ]:
import time
import urllib.request
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import sam3
import torch
from PIL import Image
from sam3 import build_sam3_image_model
from sam3.model.sam3_image_processor import Sam3Processor
from sam3.visualization_utils import plot_results

print("torch", torch.__version__, "| CUDA", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")
print("numpy", np.__version__, "| sam3 at", Path(sam3.__file__).parent)

In [ ]:
device = "cuda" if torch.cuda.is_available() else "cpu"

# TF32 matmuls (Ampere and newer) and bf16 autocast, as in the upstream examples.
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True


def inferenceContext() -> torch.autocast:
    """Autocast context for SAM3 calls (bf16 on GPU)."""
    return torch.autocast(device, dtype=torch.bfloat16, enabled=device == "cuda")

## Sample images

The upstream example images aren't shipped in the `sam3` wheel, so they're downloaded
into `data/samples/` (git-ignored). Drop your own images in `data/` and point
`imagePath` at them.

In [ ]:
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SAMPLE_DIR = REPO_ROOT / "data" / "samples"
SAMPLE_URL = "https://raw.githubusercontent.com/facebookresearch/sam3/main/assets/images/"
SAMPLE_NAMES = ["test_image.jpg", "truck.jpg", "groceries.jpg"]

SAMPLE_DIR.mkdir(parents=True, exist_ok=True)
for name in SAMPLE_NAMES:
    dest = SAMPLE_DIR / name
    if not dest.exists():
        urllib.request.urlretrieve(SAMPLE_URL + name, dest)

fig, axes = plt.subplots(1, len(SAMPLE_NAMES), figsize=(15, 4))
for ax, name in zip(axes, SAMPLE_NAMES, strict=True):
    ax.imshow(Image.open(SAMPLE_DIR / name))
    ax.set_title(name)
    ax.axis("off")
plt.show()

## Load the model

In [ ]:
t0_s = time.perf_counter()
model = build_sam3_image_model(device=device)
processor = Sam3Processor(model, device=device, confidence_threshold=0.5)
print(f"model loaded in {time.perf_counter() - t0_s:.1f} s")
print(f"{sum(p.numel() for p in model.parameters()) / 1e6:.0f} M parameters")

## Text prompt

`set_image` runs the image encoder once. Prompts are cheap after that, so you can
try many of them against the same `state`.

In [ ]:
imagePath = SAMPLE_DIR / "test_image.jpg"
image = Image.open(imagePath).convert("RGB")
imageWidth_px, imageHeight_px = image.size

with inferenceContext():
    state = processor.set_image(image)
    state = processor.set_text_prompt(state=state, prompt="shoe")

plot_results(image, state)

### What comes back

`state` is a plain dict. The detection results are:

- `masks`: bool, `(N, 1, H, W)` at the original image resolution
- `masks_logits`: float, same shape (pre-threshold)
- `boxes`: `(N, 4)` XYXY in pixels
- `scores`: `(N,)` confidence after `confidence_threshold`

In [ ]:
for key in ["masks", "masks_logits", "boxes", "scores"]:
    value = state[key]
    print(f"{key:13s} {str(tuple(value.shape)):22s} {value.dtype} on {value.device}")
print(
    "other keys:", sorted(k for k in state if k not in {"masks", "masks_logits", "boxes", "scores"})
)

## Your own visualisation (package prototype)

A numpy/matplotlib overlay that doesn't depend on `sam3.visualization_utils`. Code like this
is a candidate for `sam3Toolkit` once it settles.

In [ ]:
def resultsToNumpy(state: dict) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    """Return (masks (N, H, W) bool, boxes_XYXY_px (N, 4), scores (N,)) as numpy."""
    masks = state["masks"].squeeze(1).cpu().numpy()
    boxes_XYXY_px = state["boxes"].float().cpu().numpy()
    scores = state["scores"].float().cpu().numpy()
    return masks, boxes_XYXY_px, scores


def overlayMasks(image: Image.Image, masks: np.ndarray, alpha: float = 0.5) -> np.ndarray:
    """Blend one colour per mask onto the image; returns an (H, W, 3) float array in [0, 1]."""
    canvas = np.asarray(image, dtype=np.float32) / 255.0
    colors = plt.get_cmap("tab10").colors
    for i, mask in enumerate(masks):
        canvas[mask] = (1 - alpha) * canvas[mask] + alpha * np.asarray(colors[i % len(colors)])
    return canvas


def showResults(image: Image.Image, state: dict, title: str = "") -> None:
    masks, boxes_XYXY_px, scores = resultsToNumpy(state)
    fig, ax = plt.subplots(figsize=(10, 7))
    ax.imshow(overlayMasks(image, masks))
    for (x0, y0, x1, y1), score in zip(boxes_XYXY_px, scores, strict=True):
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, fill=False, color="white", lw=1.5))
        ax.text(
            x0,
            y0 - 4,
            f"{score:.2f}",
            color="white",
            fontsize=9,
            bbox={"facecolor": "black", "alpha": 0.6, "pad": 1, "lw": 0},
        )
    ax.set_title(f"{title} ({len(scores)} found)")
    ax.axis("off")
    plt.show()


showResults(image, state, title="shoe")

## Several prompts on one image

In [ ]:
prompts = ["person", "shoe", "shirt", "hand"]

for prompt in prompts:
    processor.reset_all_prompts(state)
    with inferenceContext():
        state = processor.set_text_prompt(state=state, prompt=prompt)
    showResults(image, state, title=prompt)

## Box prompts (visual exemplars)

`add_geometric_prompt` takes one box as **normalised (cx, cy, w, h)** in [0, 1].
`label=True` means "find things like this" and `label=False` means "not like this".

In [ ]:
def xywhToNormCxcywh(box_XYWH_px: list[float], width_px: int, height_px: int) -> list[float]:
    """Pixel (x, y, w, h) with top-left origin -> normalised (cx, cy, w, h)."""
    x, y, w, h = box_XYWH_px
    return [(x + w / 2) / width_px, (y + h / 2) / height_px, w / width_px, h / height_px]


def drawPromptBoxes(
    image: Image.Image, boxes_XYWH_px: list[list[float]], labels: list[bool]
) -> None:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(image)
    for (x, y, w, h), label in zip(boxes_XYWH_px, labels, strict=True):
        ax.add_patch(
            plt.Rectangle((x, y), w, h, fill=False, lw=2, color="lime" if label else "red")
        )
    ax.set_title("prompt boxes (green = positive, red = negative)")
    ax.axis("off")
    plt.show()


boxes_XYWH_px = [[480.0, 290.0, 110.0, 360.0], [370.0, 280.0, 115.0, 375.0]]
labels = [True, False]
drawPromptBoxes(image, boxes_XYWH_px, labels)

processor.reset_all_prompts(state)
with inferenceContext():
    for box_XYWH_px, label in zip(boxes_XYWH_px, labels, strict=True):
        normBox = xywhToNormCxcywh(box_XYWH_px, imageWidth_px, imageHeight_px)
        state = processor.add_geometric_prompt(state=state, box=normBox, label=label)
showResults(image, state, title="box prompts")

## Confidence threshold

`set_confidence_threshold` re-runs the decoder on the current prompt, so the results in
`state` update straight away (no need to set the prompt again).

In [ ]:
processor.reset_all_prompts(state)
with inferenceContext():
    state = processor.set_text_prompt(state=state, prompt="person")
    for threshold in [0.3, 0.5, 0.7]:
        state = processor.set_confidence_threshold(threshold, state)
        showResults(image, state, title=f"person @ threshold {threshold}")
    processor.set_confidence_threshold(0.5, state)

## Timing

In [ ]:
def timeIt(fn, nRepeat: int = 10) -> float:
    """Median wall time of fn() in milliseconds (CUDA-synchronised)."""
    times_ms = []
    for _ in range(nRepeat):
        if device == "cuda":
            torch.cuda.synchronize()
        t0_s = time.perf_counter()
        fn()
        if device == "cuda":
            torch.cuda.synchronize()
        times_ms.append((time.perf_counter() - t0_s) * 1e3)
    return float(np.median(times_ms))


def encodeImage() -> None:
    with inferenceContext():
        processor.set_image(image)


def promptOnly() -> None:
    processor.reset_all_prompts(state)
    with inferenceContext():
        processor.set_text_prompt(state=state, prompt="person")


encodeImage()  # warm-up
print(f"set_image:       {timeIt(encodeImage):7.1f} ms")
print(f"set_text_prompt: {timeIt(promptOnly):7.1f} ms")
if device == "cuda":
    print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 2**30:.2f} GiB")

## Your experiments

Put your own images in `data/` (git-ignored) and try them here.

In [ ]:
# myImage = Image.open(REPO_ROOT / "data" / "myImage.jpg").convert("RGB")
# with inferenceContext():
#     myState = processor.set_image(myImage)
#     myState = processor.set_text_prompt(state=myState, prompt="car")
# showResults(myImage, myState, title="car")